# Emerging Technologies Assessment

**Student:** Hasan Murtaza 


## Problem 1: Generating Random Boolean Functions

### Understanding the Requirements

The [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) works with functions that:
- Accept a fixed number of Boolean inputs (in this case: 4)
- Return a single Boolean output
- Are guaranteed to be either **constant** or **balanced**

With 4 Boolean inputs, we have 2^4 = **16 possible input combinations**.

### Constant vs Balanced Functions

**Constant functions:** Always return the same value
- Always return `False` (0), OR
- Always return `True` (1)
- Total: **2 constant functions**

**Balanced functions:** Return `True` for exactly half the inputs
- Return `True` for exactly 8 of the 16 inputs
- Return `False` for the other 8 inputs
- Total: C(16,8) = **12,870 balanced functions**

In [ ]:
# Import required libraries
import random
import numpy as np
import itertools as it

### Truth Table Representation

Any Boolean function can be represented as a truth table. For a 4-input function:

| Input (binary) | Decimal | Output |
|---------------|---------|--------|
| 0000 | 0 | f(0,0,0,0) |
| 0001 | 1 | f(0,0,0,1) |
| ... | ... | ... |
| 1111 | 15 | f(1,1,1,1) |

We can store the output column as a **tuple of length 16**, where each position corresponds to the decimal value of the binary input.

### Approach

Based on the [notes on random functions](https://github.com/ianmcloughlin/emerging-technologies/blob/main/notes/random-functions.ipynb), I will:

1. Generate a random output tuple (constant or balanced)
2. Create a closure that uses this tuple
3. The closure converts 4 Boolean inputs to an integer index
4. Return the value at that index in the tuple

This approach hides the function's internal structure, making it suitable for testing the Deutsch-Jozsa algorithm.

### Helper Function 1: Generating Random Tuples

The First helper function needed is a function that generates a random tuple representing either a constant or balanced function.

In [ ]:
def random_tuple(n):
    """
    Generate a random constant or balanced tuple of length 2**n.
    
    Parameters:
        n (int): Number of input bits
    
    Returns:
        tuple: Length 2**n, either all same value (constant) or half 0s/1s (balanced)
    """
    # Randomly choose function type (50/50 probability)
    ftype = random.choice(['constant', 'balanced'])
    
    if ftype == 'constant':
        # Choose all 0s or all 1s
        zero_or_one = random.choice([0, 1])
        # Create tuple by repeating the value 2**n times
        t = (zero_or_one,) * (2**n)
    else:  # balanced
        # Create list with half 0s, half 1s
        t = ([0] * (2**(n-1))) + ([1] * (2**(n-1)))
        # Shuffle to randomize positions
        random.shuffle(t)
        # Convert to immutable tuple
        t = tuple(t)
    
    return t

### Understanding Tuple Multiplication

The expression `(value,) * n` creates a tuple by repeating `value` n times. The comma after `value` is crucial, it makes it a tuple, not just a parenthesized expression.

In [ ]:
# Example: Create tuple of 8 zeros
zeros = (0,) * 8
print(f"Eight zeros: {zeros}")

# Example: Create tuple of 8 ones  
ones = (1,) * 8
print(f"Eight ones: {ones}") 

### Testing random_tuple



In [ ]:
# Generate some random tuples for n=4 (16 elements)
print("Random tuples for n=4:")
for i in range(5):
    t = random_tuple(4)
    # Count ones to check if constant or balanced
    ones_count = sum(t)
    if ones_count == 0 or ones_count == 16:
        ftype = "constant"
    else:
        ftype = "balanced"
    print(f"{t} - {ftype} ({ones_count} ones)")

### Helper Function 2: Converting Binary Arguments to Integer

We need to convert 4 Boolean arguments (e.g., `1, 0, 1, 1`) into an integer index (e.g., 11 in decimal = 1011 in binary).

Python's **variadic function** syntax (`*args`) allows us to accept any number of arguments. See [Real Python's guide on *args](https://realpython.com/python-boolean/) for more details.

In [ ]:
def bin_args_to_int(*args):
    """
    Convert binary arguments to an integer.
    
    Parameters:
        *args: Variable number of binary values (0/1 or False/True)
    
    Returns:
        int: Decimal representation of the binary input
    """
    # Convert any truthy values to '1', falsy to '0'
    bits = ['1' if i else '0' for i in args]
    # Join into binary string
    bits_str = ''.join(bits)
    # Convert binary string to integer (base 2)
    return int(bits_str, 2)

### Testing Binary Conversion

Verify this works correctly:

In [ ]:
# Test binary to integer conversion
print(f"Binary 0000 = {bin_args_to_int(0, 0, 0, 0)}")
print(f"Binary 0001 = {bin_args_to_int(0, 0, 0, 1)}")
print(f"Binary 1011 = {bin_args_to_int(1, 0, 1, 1)}")
print(f"Binary 1111 = {bin_args_to_int(1, 1, 1, 1)}")

### Main Function: random_constant_balanced

Now we combine everything using a **closure**. A closure is a function defined inside another function that "remembers" variables from its enclosing scope.

See [Real Python's guide on closures](https://realpython.com/python-closures/) for more information on this pattern.

In [ ]:
def random_constant_balanced():
    """
    Return a random constant or balanced function taking 4 Boolean inputs.
    
    Returns:
        function: A closure that implements the random function
    """
    # Generate random output tuple for n=4 (length 16)
    output_tuple = random_tuple(4)
    
    # Define inner function that uses the tuple
    def f(*args):
        # Ensure exactly 4 arguments
        if len(args) != 4:
            raise ValueError(f"Expected 4 arguments, got {len(args)}")
        
        # Convert binary args to index (0-15)
        index = bin_args_to_int(*args)
        
        # Return value at that index
        return output_tuple[index]
    
    return f

### Why Use a Closure?

The closure pattern is important because:
1. It **hides the output tuple** from external inspection
2. The function appears "black box" to outside code
3. This matches how the Deutsch-Jozsa algorithm works, it queries the function without knowing its internal structure

Attempting to inspect the function won't reveal whether it's constant or balanced:

In [ ]:
# Create a random function
f = random_constant_balanced()

# if we try to inspect it, we can't tell what it does.
print(f"Function object: {f}")
print(f"Function name: {f.__name__}")

### Testing the Function

Test the function by calling it with various inputs:

In [ ]:
# Create a random function
f = random_constant_balanced()

# Test with a few inputs
print("Testing random function:")
print(f"f(0,0,0,0) = {f(0,0,0,0)}")
print(f"f(0,0,0,1) = {f(0,0,0,1)}")
print(f"f(1,0,1,1) = {f(1,0,1,1)}")
print(f"f(1,1,1,1) = {f(1,1,1,1)}")

### Complete Testing: All 16 Inputs

To verify the function works correctly, let's test all 16 possible inputs and check if it's truly constant or balanced:

In [ ]:
# Create a random function
f = random_constant_balanced()

# Generate all 16 possible 4-bit inputs
all_inputs = it.product([0, 1], repeat=4)

# Test function on all inputs
results = []
print("Input (binary) | Decimal | Output")
print("-" * 35)
for inp in all_inputs:
    output = f(*inp)
    results.append(output)
    binary = ''.join(str(b) for b in inp)
    decimal = bin_args_to_int(*inp)
    print(f"{binary:14} | {decimal:7} | {output}")

# Determine function type
ones_count = sum(results)
if ones_count == 0 or ones_count == 16:
    print(f"\nFunction is CONSTANT (all {'0' if ones_count == 0 else '1'}s)")
elif ones_count == 8:
    print(f"\nFunction is BALANCED (8 ones, 8 zeros)")
else:
    print(f"\nERROR: Function is neither constant nor balanced")

### Verification: Multiple Random Functions

We should generate several random functions to confirm they're all properly constant or balanced:

In [ ]:
# Test 10 random functions
print("Testing 10 random functions:")
print("-" * 40)

for i in range(10):
    # Create random function
    f = random_constant_balanced()
    
    # Test on all inputs
    all_inputs = it.product([0, 1], repeat=4)
    results = [f(*inp) for inp in all_inputs]
    ones_count = sum(results)
    
    # Classify
    if ones_count == 0:
        ftype = "constant (all 0s)"
    elif ones_count == 16:
        ftype = "constant (all 1s)"
    elif ones_count == 8:
        ftype = "balanced"
    else:
        ftype = "ERROR"
    
    print(f"Function {i+1}: {ones_count:2} ones - {ftype}")

### Summary

We have successfully implemented `random_constant_balanced()` which:
- Returns a function taking 4 Boolean inputs
- Randomly chooses between constant and balanced functions
- Uses closures to hide the internal structure
- Properly implements the required behavior

This function will be used in Problem 2 to test classical approaches and in Problem 5 to test the quantum Deutsch-Jozsa algorithm.

## Problem 2: Classical Testing for Function Type.

### Understanding the Classical Approach

To determine if a function is constant or balanced classically, we need to query the function (call it with inputs) and observe the outputs.

**Key Analysis:**
- For a 4-bit function, there are 16 possible inputs
- **Constant function:** All 16 outputs are the same
- **Balanced function:** Exactly 8 outputs are 1, and 8 are 0

**Worst-case scenario:**
- To be 100% certain a function is **constant**, we must check all 16 inputs
- To be 100% certain a function is **balanced**, we need at most 9 inputs:
  - If first 8 outputs are all the same, function might be constant
  - The 9th output determines it: if different -> balanced, if same -> need to check all to confirm constant

**For general n-bit functions:**
- Worst case: 2^n queries (to confirm constant)
- Best case: 2 queries (to detect balanced)

### Implementation Strategy

Following the approach from the [course notes](https://github.com/ianmcloughlin/emerging-technologies/blob/main/notes/random-functions.ipynb):

1. Generate all 16 possible 4-bit inputs using `itertools.product`
2. Call function `f` with each input.
3. Collect all outputs into an array.
4. Analyze outputs:
- If all outputs are 0 OR all outputs are 1 -> constant
- If exactly 8 outputs are 1 -> balanced

In [ ]:
def determine_constant_balanced(f):
    """
    Determine if a function is constant or balanced.
    
    Uses the classical approach: test the function on all possible inputs.
    
    Parameters:
        f: Function taking 4 Boolean inputs, returning Boolean output
    
    Returns:
        str: Either "constant" or "balanced"
    """
    # Generate all 16 possible 4-bit inputs
    all_inputs = it.product([0, 1], repeat=4)
    
    # Test function on all inputs
    results = [f(*inp) for inp in all_inputs]
    
    # Convert to numpy array for analysis
    results = np.array(results)
    
    # Check if constant (all 0s or all 1s)
    if np.all(results == 0) or np.all(results == 1):
        return "constant"
    # Check if balanced (exactly 8 ones)
    elif results.sum() == 8:
        return "balanced"
    else:
        # Shouldnt happen with valid Problem 1 functions
        raise ValueError("Function is neither constant nor balanced")

### Efficiency Analysis

**Queries required:**
- Our implementation: **16 function calls** (tests all inputs)
- Guarantees 100% certainty in all cases
- Time complexity: O(2^n) where n = 4

**Maximum queries needed:**
For a 4-bit function, the maximum number of queries needed to be **100% certain** is:
- **16 queries** (worst case: must check all inputs to confirm constant)
- **9 queries minimum** to detect balanced (if first 8 are same, 9th determines it)

This will be contrasted with the quantum approach in Problems 4 and 5, where [Deutschs algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm) can determine the answer with just **1 query**.

In [ ]:
# Test with random functions from Problem 1
print("Testing determine_constant_balanced:\n")

for i in range(10):
    # Generate random function
    f = random_constant_balanced()
    
    # Determine its type
    result = determine_constant_balanced(f)
    
    print(f"Function {i+1}: {result}")

### Verification: Manual Cross-Check

To verify our function works correctly, manually check one function by examining all its outputs:

In [ ]:
# Create a random function
f = random_constant_balanced()

# Use our function to determine type
determined_type = determine_constant_balanced(f)

# Manually verify
all_inputs = it.product([0, 1], repeat=4)
outputs = [f(*inp) for inp in all_inputs]
ones_count = sum(outputs)

# Manual classification
if ones_count == 0 or ones_count == 16:
    manual_type = "constant"
else:
    manual_type = "balanced"

print(f"Determined type: {determined_type}")
print(f"Manual check: {manual_type}")
print(f"Ones count: {ones_count}/16")
print(f"Match: {determined_type == manual_type}")